# Coffee Shop Demo (Google Maps / OSM)

Denne notebook viser et kort (Google Maps hvis du har sat `GOOGLE_MAPS_API_KEY`, ellers OpenStreetMap) og sender Coffee Shop-demoens MQTT-beskeder.

**Brug:**
1. Kør alle celler 1–4 for at sætte kort og MQTT op.
2. Kør celle 5 med `publish_demo(...)` for at starte demoen.
3. Kør celle 6 med `stop_demo()` når du vil stoppe.

In [9]:
# Imports og konfiguration
import os
import json
import time
import random
from IPython.display import display
import importlib

import simulated_city.maplibre_live as maplibre_live
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector

importlib.reload(maplibre_live)
LiveMapLibreMap = maplibre_live.LiveMapLibreMap

cfg = load_config()
GMAP_KEY = os.environ.get('GOOGLE_MAPS_API_KEY') or getattr(cfg, 'google_maps_api_key', None)

# Vælg Google Maps tiles hvis API-nøgle findes, ellers OSM
if GMAP_KEY:
    tiles_url = f"https://mt1.google.com/vt/lyrs=m&x={'{x}'}&y={'{y}'}&z={'{z}'}&key={GMAP_KEY}"
else:
    tiles_url = 'https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png'

CITY_HALL_LNGLAT = (12.5683, 55.6761)  # København Rådhus

In [10]:
# Opret og vis kort
m = LiveMapLibreMap(center=CITY_HALL_LNGLAT, zoom=16.5, height='700px', tiles_url=tiles_url)
m.add_3d_buildings()
display(m)
print('Using basemap:', 'Google Maps' if GMAP_KEY else 'OpenStreetMap')

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Using basemap: OpenStreetMap


In [11]:
# Forbind til MQTT-broker fra config.yaml
connector = MqttConnector(cfg.mqtt, client_id_suffix='coffee-notebook')
connector.connect()
if not connector.wait_for_connection(timeout=10.0):
    raise RuntimeError('Failed to connect to MQTT broker')

print('✓ Connected to MQTT broker')
print(f'  Broker: {cfg.mqtt.host}:{cfg.mqtt.port}')

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


✓ Connected to MQTT broker
  Broker: 127.0.0.1:1883


In [ ]:
# Demo: 3 personer og 4 coffee shops med regn-cyklus
def publish_demo(n_persons=3, steps=200, delay=0.3):
    """Publiserer positionsdata for n_persons til 'persons/{name}/location' og regn-tilstand til 'weather/rain'.
    Hver 20. sekund starter regn (personernes farve bliver rød), efter 10 sekunders regn bliver det lyst igen.
    Kør denne funktion manuelt for at starte demoen."""
    # Fire coffee shops (lng, lat)
    COFFEE_SHOPS = [
        (12.5678, 55.6765),
        (12.5687, 55.6759),
        (12.5690, 55.6763),
        (12.5680, 55.6760),
    ]

    persons = {}
    for i in range(n_persons):
        name = f'coffee-{i+1}'
        color = '#{:06x}'.format(random.randint(0, 0xFFFFFF))
        shop = random.choice(COFFEE_SHOPS)
        lng = shop[0] + random.uniform(-0.0005, 0.0005)
        lat = shop[1] + random.uniform(-0.0005, 0.0005)
        persons[name] = {'lng': lng, 'lat': lat, 'color': color}

    # Regn-tilstand: 20 sekunder tør, 10 sekunder regn (gentaget)
    state = 'dry'
    state_started = time.time()
    raining = False

    for _ in range(steps):
        now = time.time()
        elapsed = now - state_started

        # Opdatér regn-tilstand efter tidsreglerne
        if state == 'dry' and elapsed >= 20:
            state = 'rain'
            state_started = now
            raining = True
            weather_payload = json.dumps({'raining': True, 'timestamp': now})
            connector.client.publish('weather/rain', weather_payload, qos=0)
        elif state == 'rain' and elapsed >= 10:
            state = 'dry'
            state_started = now
            raining = False
            weather_payload = json.dumps({'raining': False, 'timestamp': now})
            connector.client.publish('weather/rain', weather_payload, qos=0)

        for name, st in persons.items():
            # Lille random walk omkring nuværende position
            st['lng'] += random.uniform(-0.00025, 0.00025)
            st['lat'] += random.uniform(-0.00025, 0.00025)
            current_color = '#ff0000' if raining else st['color']
            payload = {
                'lng': float(st['lng']),
                'lat': float(st['lat']),
                'color': current_color,
                'name': name,
                'timestamp': now,
            }
            topic = f'persons/{name}/location'
            connector.client.publish(topic, json.dumps(payload), qos=0)

        time.sleep(delay)

    print('Demo finished')

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


In [16]:
# Stop helper
def stop_demo():
    """Afbryder MQTT-forbindelsen fra denne notebook."""
    try:
        connector.disconnect()
    except Exception:
        pass
    print('Disconnected from MQTT')

# Eksempel: kør publish_demo(n_persons=3, steps=200, delay=0.3) for at starte demoen.

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
